In [ ]:
%pip install pymupdf langchain-community sentence-transformers faiss-cpu rank_bm25

In [ ]:
import time
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder

# --- 1. LOAD AND PARSE THE PDFs ---
print("Loading Annual Reports...")

# Load your specific uploaded files
jkh_loader = PyMuPDFLoader("jkh24:25.pdf")
vone_loader = PyMuPDFLoader("vone24:25.pdf")

documents = jkh_loader.load() + vone_loader.load()
print(f"Loaded {len(documents)} total pages.")

# --- 2. CHUNK THE DOCUMENTS ---
# We split the massive PDFs into smaller, 1000-character chunks with overlap
# so the semantic meaning is preserved but it fits in the LLM's context window.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} searchable chunks.")

# --- 3. BUILD THE HYBRID RETRIEVER ---


print("Initializing Embedding Model, FAISS, and BM25...")
# Using BGE-M3 for financial domain embeddings
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# Stage 1A: Dense Vector Retrieval (FAISS)
vectorstore = FAISS.from_documents(chunks, embedding_model)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Stage 1B: Sparse Lexical Retrieval (BM25)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

# Stage 1C: Combine into Ensemble Retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever], 
    weights=[0.5, 0.5] 
)

# --- 4. INITIALIZE CROSS-ENCODER RERANKER ---
print("Loading Cross-Encoder...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def query_financial_reports(query, top_k=3):
    """
    Searches the John Keells and Vallibel One reports for the answer.
    """
    print(f"\n--- Searching for: '{query}' ---")
    
    # 1. Fetch broad recall set
    initial_docs = hybrid_retriever.invoke(query)
    
    # Deduplicate
    unique_docs = {doc.page_content: doc for doc in initial_docs}.values()
    docs_list = list(unique_docs)
    
    # 2. Rerank for mathematical/financial precision
    pairs = [[query, doc.page_content] for doc in docs_list]
    scores = reranker.predict(pairs)
    
    # Sort by confidence score
    scored_docs = sorted(zip(docs_list, scores), key=lambda x: x[1], reverse=True)
    
    # Extract the top results
    print("\n[Top Retrieved Contexts After Reranking]:")
    best_chunks = []
    for i, (doc, score) in enumerate(scored_docs[:top_k]):
        print(f"\nRank {i+1} (Score: {score:.2f}) from page {doc.metadata.get('page', 'Unknown')}:")
        print(f"...{doc.page_content[:200]}...") # Print a snippet to verify
        best_chunks.append(doc.page_content)
        
    return "\n---\n".join(best_chunks)

# --- 5. TEST IT! ---
# Let's ask a question specifically related to the John Keells text!
test_query_1 = "What major integrated resort project was launched by John Keells Holdings this year?"
context_1 = query_financial_reports(test_query_1)

test_query_2 = "Who is the Chairman and Co-Chairman of Vallibel One PLC?"
context_2 = query_financial_reports(test_query_2)